## ChatBot with message summarization

### Goal

- Now, let's take it one step further!
- Rather than just trimming or filtering messages, we'll show how to use LLMs to produce a running summary of the conversation.
- This allows us to retain a compressed representation of the full conversation, rather than just removing it with trimming or filtering.
- We'll incorporate this summarization into a simple Chatbot.
- And we'll equip that Chatbot with memory, supporting long-running conversations without incurring high token cost / latency.


In [ ]:
from langchain_ollama import ChatOllama




In [10]:
ollama = ChatOllama(model= "qwen2.5:7b")

In [3]:
from langgraph.graph import MessagesState

In addition to the built-in messages key, we'll now include a custom key (summary).

In [12]:
class State(MessagesState):
    summary: str

In [6]:
from langchain_core.messages import SystemMessage, HumanMessage, RemoveMessage

In [13]:
def call_model(state:State):

    # Get summary if exists

    summary = state.get(summary, "")

    if summary:
        # Add summary to system message
        system_message = f"Summary of conversation earlier: {summary}"
        # Append summary to any newer messages
        messages = [SystemMessage(content=system_message)] + state["messages"]
    
    else:
        messages = state["messages"]
    
    response = ollama.invoke(messages)

    return {
        "messages": response
    }
    




Note, here we'll use RemoveMessage to filter our state after we've produced the summary.

In [15]:
def summarize_converstation(state:State):
    summary = state.get("summary", "")

    if summary:
        # A summary already exists
        summary_message = (
            f"This is summary of the conversation to date: {summary}\n\n"
            "Extend the summary by taking into account the new messages above:"
        )
    
    else:
        summary_message = "Create a summary of the conversation above:"
    # Add prompt to our history
    messages = state["messages"] + [HumanMessage(content=summary_message)]
    response = ollama.invoke(messages)

    # Delete all but the 2 most recent messages
    delete_messages = [RemoveMessage(id=m.id) for m in state["messages"][:-2]]

    return {"summary": response.content, "messages": delete_messages}







In [ ]:

from langgraph.graph import END
from typing_extensions import Literal
# Determine whether to end or summarize the conversation
def should_continue(state: State) -> Literal ["summarize_conversation",END]:
    """Return the next node to execute."""
    messages = state["messages"]
    
    # If there are more than six messages, then we summarize the conversation
    if len(messages) > 6:
        return "summarize_conversation"
    
    # Otherwise we can just end
    return END